In [ ]:
# Load VAE and test representation with trafalgar
from vae import TacticVAE as vae, evaluate_vae
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset


# Define details
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define model
checkpoint_path = "artifacts/checkpoints/vae_best.pt"
model_checkpoint = torch.load(checkpoint_path, map_location=device)
model_config = model_checkpoint.get("model_config", {"input_dim": 400, "hidden_dim": 64, "latent_dim": 8})
vae_model = vae(
    input_dim=model_config["input_dim"],
    hidden_dim=model_config["hidden_dim"],
    latent_dim=model_config["latent_dim"],
).to(device)
vae_model.load_state_dict(model_checkpoint["model_state_dict"])
vae_model.eval()

# Load data
X_ood = np.load("data/processed/trafalgar_trajectories.npy")
X_ood_tensor = torch.as_tensor(X_ood, dtype=torch.float32)
ood_loader = DataLoader(TensorDataset(X_ood_tensor), batch_size=32, shuffle=False)
ood_loss = evaluate_vae(vae_model, device, ood_loader)
print(f"OOD loss: {ood_loss}")